## EDA: Piezometria — `piezo_tejo_loc_zvt.csv`

Caminho do dataset:
`/Users/diogopinto/Documents/Usar/git_clep/clepsydra_isa/EDA /data/piezo_tejo_loc_zvt.csv`

Objetivo: caracterizar a série (cobertura temporal, estatísticas, outliers, sazonalidade), localização dos pontos e export de resumos — tudo dentro de `EDA`.


### 1) Setup de bibliotecas e configuração de paths


In [9]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

DATA_FILE = Path("/Users/diogopinto/Documents/Usar/git_clep/clepsydra_isa/EDA /data/piezo_tejo_loc_zvt.csv")
assert DATA_FILE.exists(), f"CSV not found: {DATA_FILE}"

pd.options.display.float_format = "{:.3f}".format
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print(f"Using file: {DATA_FILE}")


Using file: /Users/diogopinto/Documents/Usar/git_clep/clepsydra_isa/EDA /data/piezo_tejo_loc_zvt.csv


### 2) Leitura do CSV e preparação de colunas

#### 2.1 Explicação do código
- `parse_dates` para `data`.
- Tipos otimizados.
- Renomeação para nomes explícitos.
- Colunas derivadas: `year`, `month`, `site_id` (coordenadas `y_m`/`x_m`) e `site_label` = `well_code | y_m_x_m` para identificação humana nas tabelas e gráficos.


In [10]:
parse_dates = ["data"]
df = pd.read_csv(
    DATA_FILE,
    parse_dates=parse_dates,
    dtype={
        "id": "int32",
        "codigo": "string",
        "nivel_piezometrico": "float32",
        "profundidade_nivel_agua": "float32",
        "coord_x_m": "float32",
        "coord_y_m": "float32",
        "altitude_m": "float32",
        "sistema_aquifero": "string",
        "estado": "string",
        "freguesia": "string",
    },
)

# Ordenar e renomear
df = df.sort_values("data").reset_index(drop=True)
rename_map = {
    "data": "date",
    "codigo": "well_code",
    "nivel_piezometrico": "piezometric_level_m",
    "profundidade_nivel_agua": "water_depth_m",
    "coord_x_m": "x_m",
    "coord_y_m": "y_m",
    "altitude_m": "altitude_m",
}
df = df.rename(columns=rename_map)

# Derivadas
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
# site_id com base em coordenadas (y, x) arredondadas
df["site_id"] = df["y_m"].round(2).astype(str) + "_" + df["x_m"].round(2).astype(str)
# label legível combinando código do furo e coordenadas
df["site_label"] = df["well_code"].astype(str) + " | " + df["site_id"]

print(df.head())
df.info()


   id                      date well_code  piezometric_level_m  water_depth_m        x_m        y_m  altitude_m  \
0   4 1974-10-01 00:00:00+00:00    405/52               26.670          3.330 148760.000 218110.000      30.000   
1  26 1977-09-01 00:00:00+00:00     420/8               20.110         23.890 152403.406 204253.406      44.000   
2  27 1977-09-01 00:00:00+00:00     420/9               20.390         23.610 152404.094 204254.594      44.000   
3  29 1977-09-09 00:00:00+00:00     420/8               20.110         23.890 152403.406 204253.406      44.000   
4  31 1977-10-01 00:00:00+00:00     420/9               20.370         23.630 152404.094 204254.594      44.000   

                            sistema_aquifero estado  freguesia                     created_at  year  month  \
0  T3 - BACIA DO TEJO-SADO / MARGEM ESQUERDA   <NA>  BENAVENTE  2025-07-17 14:16:33.426554+00  1974     10   
1  T3 - BACIA DO TEJO-SADO / MARGEM ESQUERDA   <NA>      CANHA  2025-07-17 14:16:33.42655

### 3) Cobertura temporal, dias em falta, duplicados e estatísticas

#### 3.1 Explicação do código
- Frequência mensal (parece mensal pela amostra; ajusta facilmente para diária se necessário).
- `missing_dates` por local e para o total.
- `describe` para `piezometric_level_m` e `water_depth_m`.


In [11]:
from pandas.tseries.frequencies import to_offset

start_date = df["date"].min()
end_date = df["date"].max()
expected_freq = to_offset("1MS")  # monthly start
print(f"Start: {start_date:%Y-%m-%d}  End: {end_date:%Y-%m-%d}")

numeric_cols = ["piezometric_level_m", "water_depth_m"]
display(df[numeric_cols].describe(percentiles=[0.01, 0.05, 0.95, 0.99]).T)

dups = df[df.duplicated(["date", "site_id"], keep=False)]
print(f"Duplicate timestamps (per site): {dups.shape[0]}")
display(dups.head())


Start: 1974-10-01  End: 2025-01-16


,count,mean,std,min,1%,5%,50%,95%,99%,max
piezometric_level_m,17308.000,13.222,14.946,-31.190,-12.625,-2.250,9.650,37.450,53.916,81.210
water_depth_m,17308.000,20.029,16.689,-2.120,0.600,1.270,13.640,47.940,55.400,134.390


Duplicate timestamps (per site): 0


,id,date,well_code,piezometric_level_m,water_depth_m,x_m,y_m,altitude_m,sistema_aquifero,estado,freguesia,created_at,year,month,site_id,site_label


### 3.2) Frequência temporal por ponto e lacunas na série

#### 3.2.1 Explicação do código
- Para cada `site_id` é inferida a frequência dominante (horária, diária, mensal) a partir do intervalo mais comum entre observações.
- Constrói-se o calendário esperado entre o primeiro e o último registo e contam‑se os períodos em falta.
- O resumo é anexado à tabela de localizações, adicionando: `freq_label`, `missing_count` e `missing_first_5`.


In [12]:
from collections import Counter

# Função auxiliar para inferir frequência dominante

def infer_frequency(ts: pd.Series) -> str:
    if ts.size < 2:
        return "unknown"
    diffs = ts.sort_values().diff().dropna()
    if diffs.empty:
        return "unknown"
    # Mapa simples de rótulos por proximidade em dias/horas
    median = diffs.median()
    days = median / pd.Timedelta(days=1)
    hours = median / pd.Timedelta(hours=1)
    if abs(hours - 1) < 0.1:
        return "hourly"
    if abs(days - 1) < 0.2:
        return "daily"
    if 0.8 <= days <= 1.2:
        return "daily"
    # mensal: cerca de 28–31 dias
    if 27 <= days <= 31:
        return "monthly"
    # fallback: melhor aproximação
    if days > 1 and days < 10:
        return f"{days:.1f}d"
    if hours < 24:
        return f"{hours:.1f}h"
    return f"{days:.1f}d"


def compute_missing_for_site(df_site: pd.DataFrame, freq_label: str) -> tuple[int, list]:
    if df_site.empty:
        return 0, []
    start = df_site["date"].min()
    end = df_site["date"].max()
    if freq_label == "hourly":
        freq = "H"
    elif freq_label == "daily":
        freq = "D"
    else:
        # assume monthly por defeito
        freq = "MS"
    full_range = pd.date_range(start=start, end=end, freq=freq)
    missing = full_range.difference(df_site["date"].sort_values().unique())
    return int(missing.size), [str(x) for x in list(missing[:5])]

# Calcular por site
site_freq_rows = []
for site_id, df_s in df.groupby("site_id"):
    freq_label = infer_frequency(df_s["date"])
    missing_count, missing_first_5 = compute_missing_for_site(df_s, freq_label)
    site_label = df_s["site_label"].iloc[0]
    site_freq_rows.append({
        "site_id": site_id,
        "site_label": site_label,
        "freq_label": freq_label,
        "missing_count": missing_count,
        "missing_first_5": ", ".join(missing_first_5),
    })

site_frequency_summary = pd.DataFrame(site_freq_rows)

display(site_frequency_summary.head())


,site_id,site_label,freq_label,missing_count,missing_first_5
0,181651.8_127393.1,443/924 | 181651.8_127393.1,daily,8632,"1999-12-02 00:00:00+00:00, 1999-12-03 00:00:00..."
1,182086.3_140865.0,444/85 | 182086.3_140865.0,monthly,208,"2001-07-01 00:00:00+00:00, 2001-08-01 00:00:00..."
2,183790.2_146361.6,444/317 | 183790.2_146361.6,daily,8668,"1999-12-02 00:00:00+00:00, 1999-12-03 00:00:00..."
3,183830.0_146440.0,444/318 | 183830.0_146440.0,31.5d,76,"2005-06-01 00:00:00+00:00, 2005-07-01 00:00:00..."
4,190187.4_133281.5,432/855 | 190187.4_133281.5,monthly,256,"2003-06-01 00:00:00+00:00, 2003-07-01 00:00:00..."


### 4) Deteção de outliers (IQR e Z-score)


In [13]:
def detect_outliers_iqr(series: pd.Series, factor: float = 1.5) -> pd.Series:
    q1 = np.nanpercentile(series, 25)
    q3 = np.nanpercentile(series, 75)
    iqr = q3 - q1
    lower = q1 - factor * iqr
    upper = q3 + factor * iqr
    return (series < lower) | (series > upper)


def detect_outliers_zscore(series: pd.Series, threshold: float = 3.0) -> pd.Series:
    mu = np.nanmean(series)
    sigma = np.nanstd(series)
    if sigma == 0 or np.isnan(sigma):
        return pd.Series(False, index=series.index)
    z = (series - mu) / sigma
    return z.abs() > threshold

for col in numeric_cols:
    flags = detect_outliers_iqr(df[col]) | detect_outliers_zscore(df[col])
    df[f"is_outlier_{col}"] = flags
    print(f"{col}: {flags.sum()} possíveis outliers")

display(df.loc[df.filter(like="is_outlier_").any(axis=1), ["date", "site_id"] + numeric_cols].head(10))


piezometric_level_m: 395 possíveis outliers
water_depth_m: 97 possíveis outliers


,date,site_id,piezometric_level_m,water_depth_m
99,1978-11-01 00:00:00+00:00,193570.0_131190.0,-11.880,63.880
117,1979-01-01 00:00:00+00:00,193570.0_131190.0,-11.290,63.290
139,1979-04-01 00:00:00+00:00,199930.2_156971.4,53.470,6.530
152,1979-05-01 00:00:00+00:00,199930.2_156971.4,53.190,6.810
167,1979-06-01 00:00:00+00:00,199930.2_156971.4,53.070,6.930
189,1979-07-01 00:00:00+00:00,199930.2_156971.4,52.920,7.080
190,1979-07-01 00:00:00+00:00,193570.0_131190.0,-12.440,64.440
205,1979-08-01 00:00:00+00:00,199930.2_156971.4,52.700,7.300
208,1979-08-01 00:00:00+00:00,193570.0_131190.0,-12.730,64.730
223,1979-09-01 00:00:00+00:00,199930.2_156971.4,52.580,7.420


### 5B) Gráficos por ponto específico (com seletor)
Use o dropdown para escolher um `site_id` e visualizar apenas esse ponto.


In [18]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Mapear de site_id -> label legível (well_code | coords)
site_to_label = (
    df.groupby("site_id")["site_label"].first().to_dict()
)

sites = sorted(df["site_id"].unique())
print(f"Total sites: {len(sites)}")

# Mostrar dropdown com label, mas manter valor interno como site_id
options = [(site_to_label[s], s) for s in sites]
out = widgets.Output()
site_dropdown = widgets.Dropdown(options=options, value=sites[0], description="site:")

def plot_site(site_id: str) -> None:
    df_s = df[df["site_id"] == site_id]
    label = site_to_label.get(site_id, site_id)
    fig, ax = plt.subplots(1, 1, figsize=(14, 6))

    # Quebrar ligações quando o intervalo entre pontos > 2 meses (~62 dias)
    gap_threshold = pd.Timedelta(days=62)
    seg_id = (df_s["date"].diff() > gap_threshold).cumsum()

    color = "tab:blue"
    added_label = False
    for _, g in df_s.groupby(seg_id):
        if len(g) >= 2:
            ax.plot(g["date"], g["piezometric_level_m"],
                    label=(f"{label} · piezometric_level_m" if not added_label else None),
                    color=color)
            added_label = True
        else:
            # ponto isolado: mostrar apenas como scatter
            ax.scatter(g["date"], g["piezometric_level_m"], color=color, s=12)

    # Outliers em destaque
    if f"is_outlier_piezometric_level_m" in df_s.columns:
        mask = df_s["is_outlier_piezometric_level_m"]
        ax.scatter(
            df_s.loc[mask, "date"], df_s.loc[mask, "piezometric_level_m"],
            s=28, color="black", label="outlier"
        )
    ax.legend(); ax.set_xlabel("date"); ax.set_ylabel("piezometric_level_m [m]")
    ax.set_title(f"Piezometric level over time ({label}) — gaps > 2 months not connected")
    ax.grid(True, alpha=0.2)
    plt.tight_layout(); plt.show()


def on_change(ch):
    if ch["name"] == "value":
        with out:
            clear_output(wait=True)
            plot_site(ch["new"]) 

site_dropdown.observe(on_change, names="value")
display(site_dropdown)
with out:
    plot_site(site_dropdown.value)
display(out)


Total sites: 73


Dropdown(description='site:', options=(('443/924 | 181651.8_127393.1', '181651.8_127393.1'), ('444/85 | 182086…

Output()

### 6) Localização dos pontos (resumo)
- Número de locais únicos.
- Tabela por local com nº de observações e intervalo temporal.


In [15]:
unique_points = df[["y_m", "x_m"]].drop_duplicates().shape[0]
print(f"Unique measurement locations: {unique_points}")

location_summary = (
    df.groupby(["site_id", "site_label", "y_m", "x_m"]).agg(
        n_obs=("date", "count"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        mean_piezometric_level=("piezometric_level_m", "mean"),
    ).reset_index()
)

# Juntar frequência e lacunas por ponto
location_summary = location_summary.merge(
    site_frequency_summary, on=["site_id", "site_label"], how="left"
)

display(location_summary.head())


Unique measurement locations: 73


,site_id,site_label,y_m,x_m,n_obs,first_date,last_date,mean_piezometric_level,freq_label,missing_count,missing_first_5
0,181651.8_127393.1,443/924 | 181651.8_127393.1,181651.797,127393.102,534,1999-12-01 00:00:00+00:00,2025-01-03 00:00:00+00:00,6.844,daily,8632,"1999-12-02 00:00:00+00:00, 1999-12-03 00:00:00..."
1,182086.3_140865.0,444/85 | 182086.3_140865.0,182086.297,140865.000,155,2001-01-01 00:00:00+00:00,2019-03-01 00:00:00+00:00,29.440,monthly,208,"2001-07-01 00:00:00+00:00, 2001-08-01 00:00:00..."
2,183790.2_146361.6,444/317 | 183790.2_146361.6,183790.203,146361.594,505,1999-12-01 00:00:00+00:00,2025-01-10 00:00:00+00:00,16.494,daily,8668,"1999-12-02 00:00:00+00:00, 1999-12-03 00:00:00..."
3,183830.0_146440.0,444/318 | 183830.0_146440.0,183830.000,146440.000,59,2005-05-23 00:00:00+00:00,2011-11-07 00:00:00+00:00,21.084,31.5d,76,"2005-06-01 00:00:00+00:00, 2005-07-01 00:00:00..."
4,190187.4_133281.5,432/855 | 190187.4_133281.5,190187.406,133281.500,178,2003-05-14 00:00:00+00:00,2025-01-10 00:00:00+00:00,11.448,monthly,256,"2003-06-01 00:00:00+00:00, 2003-07-01 00:00:00..."


### 7) Export de tabelas resumo (dentro de EDA)

#### 7.1 Explicação do código
- Estatísticas resumo das variáveis principais.
- Médias mensais do nível piezométrico.
- Export para `EDA /scripts/piezo/resources/`.


In [16]:
summary_stats = df[["piezometric_level_m", "water_depth_m"]].describe().T
monthly_means = df.groupby(["year", "month"]) [["piezometric_level_m"]].mean().reset_index()

# Adiciona nº de pontos e frequência global (moda das frequências) ao resumo
summary_stats.loc["meta_unique_points", ["count", "mean"]] = [unique_points, np.nan]
# frequência dominante no dataset
freq_mode = site_frequency_summary["freq_label"].mode().iat[0] if not site_frequency_summary.empty else "unknown"
summary_stats.loc["meta_freq_mode", ["count", "mean"]] = [np.nan, np.nan]
summary_stats.loc["meta_freq_mode", "std"] = freq_mode

from pathlib import Path
out_dir = Path("/Users/diogopinto/Documents/Usar/git_clep/clepsydra_isa/EDA /scripts/piezo/resources")
out_dir.mkdir(parents=True, exist_ok=True)
summary_stats.to_csv(out_dir / "piezo_summary_stats.csv")
monthly_means.to_csv(out_dir / "piezo_monthly_means.csv", index=False)
location_summary.to_csv(out_dir / "piezo_location_summary.csv", index=False)
site_frequency_summary.to_csv(out_dir / "piezo_frequency_gaps_by_site.csv", index=False)

print(f"Saved summaries in: {out_dir}")


Saved summaries in: /Users/diogopinto/Documents/Usar/git_clep/clepsydra_isa/EDA /scripts/piezo/resources


/var/folders/f9/slpppqbj1fs9tjk3hnc6r70c0000gn/T/ipykernel_62673/2430730053.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'monthly' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  summary_stats.loc["meta_freq_mode", "std"] = freq_mode
